# 11 - DSP Blocks and Correlators

**Objective:** Use FIR filters, DDS tuning, and hardware correlators for advanced signal processing. This notebook covers:
- Configuring and applying FIR filters
- Dynamic DDS frequency tuning during sequences
- Using hardware correlators for multi-channel measurements
- Generating multi-tone signals and performing spectrum analysis

## 1. Setup

In [ ]:
# Standard imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import logging
from scipy import signal

from qick import *
from qick.asm_v2 import AveragerProgramV2

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-8s [%(filename)s:%(lineno)d] %(message)s')

# Connect to the board
BITSTREAM_PATH = '/path/to/your/firmware.bit'  # <--- CHANGE THIS
soc = QickSoc(BITSTREAM_PATH)
soccfg = soc

# Define hardware channels
GEN_CH = 0
RO_CH = 0

print("Ready")

## 2. FIR Filters Overview

Finite Impulse Response (FIR) filters are implemented in the FPGA fabric for real-time signal processing. They can be used for:
- Pulse shaping (e.g., Gaussian, raised cosine)
- Band limiting and anti-aliasing
- Matched filtering for optimal SNR
- Removing unwanted frequency components

## 3. Configuring an FIR Filter

FIR filters are applied to DAC outputs. Let's design and apply a low-pass filter.

In [ ]:
def design_lowpass_filter(cutoff_mhz, sample_rate_mhz=100, num_taps=31):
    """
    Design a simple low-pass FIR filter.
    
    Parameters:
    - cutoff_mhz: Cutoff frequency in MHz
    - sample_rate_mhz: DAC sample rate in MHz (typically 100 for 1ns resolution)
    - num_taps: Number of filter taps (must be odd, max 255 for QICK)
    """
    # Normalized cutoff frequency (Nyquist = 0.5)
    nyquist = sample_rate_mhz / 2
    normalized_cutoff = cutoff_mhz / nyquist
    
    # Design filter using Kaiser window
    taps = signal.firwin(num_taps, normalized_cutoff, window='hamming')
    
    # Scale to QICK's 16-bit integer representation
    # QICK expects coefficients in Q1.15 format (range -32768 to 32767)
    max_tap = np.max(np.abs(taps))
    taps_scaled = (taps / max_tap * 32767).astype(int)
    
    return taps_scaled

# Design a filter with 20 MHz cutoff
cutoff_freq = 20  # MHz
filter_taps = design_lowpass_filter(cutoff_freq, num_taps=31)

# Visualize the filter response
freq_response = np.abs(np.fft.rfft(filter_taps, n=1024))
freqs = np.linspace(0, 50, len(freq_response))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.stem(filter_taps, basefmt=' ')
plt.xlabel('Tap Index')
plt.ylabel('Coefficient Value')
plt.title(f'FIR Filter Taps (31 taps, {cutoff_freq} MHz cutoff)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(freqs, 20 * np.log10(freq_response / np.max(freq_response) + 1e-10))
plt.axvline(x=cutoff_freq, color='r', linestyle='--', label=f'{cutoff_freq} MHz cutoff')
plt.xlabel('Frequency (MHz)')
plt.ylabel('Gain (dB)')
plt.title('Filter Frequency Response')
plt.legend()
plt.grid(True)
plt.ylim(-80, 5)
plt.tight_layout()
plt.show()

print(f"Filter coefficients: {filter_taps[:10]}...")
print(f"Number of taps: {len(filter_taps)}")

In [ ]:
class FIRFilterProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Apply FIR filter to the generator channel
        # The filter is applied to the baseband signal before IQ modulation
        self.set_fir_filter(
            ch=cfg['gen_ch'],
            coeffs=cfg['filter_coeffs'],
            gain=cfg.get('filter_gain', 1.0)  # Optional gain compensation
        )

        # Define a test pulse (square pulse to see filtering effect)
        self.add_pulse(
            ch=cfg['gen_ch'], name="test_pulse",
            style="const",
            freq=cfg['freq'], length=cfg['pulse_len'],
            phase=0, gain=cfg['gain']
        )

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="my_ro", t=0)

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0)

# Configuration with filter
config_fir = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq': 100.0,
    'pulse_len': 0.5,
    'gain': 0.5,
    'trig_time': 0.05,
    'ro_len': 0.8,
    'filter_coeffs': filter_taps.tolist(),
    'filter_gain': 1.0
}

print("FIR filter configured for DAC channel.")
print("The filter will shape the output pulse spectrum.")
print("\nNote: FIR filters are applied in real-time on the FPGA.")
print("To see the effect, compare the pulse shape with and without the filter.")

## 4. Dynamic DDS Tuning

The Direct Digital Synthesizer (DDS) generates the carrier frequency. You can change the DDS frequency mid-sequence, enabling frequency sweeps and multi-frequency experiments.

In [ ]:
class DynamicDDSProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        self.add_pulse(ch=cfg['gen_ch'], name="test_pulse",
                       style="const", length=0.1, phase=0, gain=cfg['gain'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq'], gen_ch=cfg['gen_ch'])
        self.send_readoutconfig(ch=cfg['ro_ch'], name="my_ro", t=0)

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True)
        
        # Pulse at frequency 1
        self.set_dds_freq(ch=cfg['gen_ch'], freq=cfg['freq1'], t=0)
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0.05, tag='f1')
        
        # Change frequency mid-sequence
        self.set_dds_freq(ch=cfg['gen_ch'], freq=cfg['freq2'], t=0.25)
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0.3, tag='f2')
        
        # Change to frequency 3
        self.set_dds_freq(ch=cfg['gen_ch'], freq=cfg['freq3'], t=0.55)
        self.pulse(ch=cfg['gen_ch'], name="test_pulse", t=0.6, tag='f3')

config_dds = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq1': 90.0,   # MHz
    'freq2': 100.0,  # MHz
    'freq3': 110.0,  # MHz
    'gain': 0.5,
    'trig_time': 0.02,
    'ro_len': 0.9
}

prog = DynamicDDSProgram(soccfg, reps=1, final_delay=0.5, cfg=config_dds)
print("=== Dynamic DDS Program ===")
print("The DDS frequency changes three times during the sequence:")
print(f"  - 0.05 µs: {config_dds['freq1']} MHz")
print(f"  - 0.30 µs: {config_dds['freq2']} MHz")
print(f"  - 0.60 µs: {config_dds['freq3']} MHz")
print("\nNote: You can use this for multi-frequency experiments like:")
print("  - Frequency-selective qubit control")
print("  - Multi-tone readout")
print("  - Off-resonant driving experiments")

## 5. Hardware Correlators

The QICK includes hardware correlators that can compute cross-correlations between channels in real-time. This is useful for:
- Multi-qubit readout
- Noise correlation measurements
- Time-of-flight experiments
- Quantum state tomography

In [ ]:
class HardwareCorrelatorProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        self.declare_gen(ch=cfg['gen_ch'], nqz=1)
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])
        
        # Enable hardware correlator
        # This correlates two ADC channels
        self.enable_correlator(
            ch0=cfg['corr_ch0'],   # First ADC channel
            ch1=cfg['corr_ch1'],   # Second ADC channel
            length=cfg['corr_len']  # Correlation length in samples
        )

        # Define pulses on two generator channels
        self.add_pulse(ch=cfg['gen_ch'], name="pulse1",
                       style="const", freq=cfg['freq1'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain'])
        
        self.add_pulse(ch=cfg['gen_ch2'], name="pulse2",
                       style="const", freq=cfg['freq2'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="ro1",
                               freq=cfg['freq1'], gen_ch=cfg['gen_ch'])
        self.add_readoutconfig(ch=cfg['ro_ch2'], name="ro2",
                               freq=cfg['freq2'], gen_ch=cfg['gen_ch2'])

    def _body(self, cfg):
        self.pulse(ch=cfg['gen_ch'], name="pulse1", t=0)
        self.pulse(ch=cfg['gen_ch2'], name="pulse2", t=cfg['delay'])
        
        self.trigger(ros=[cfg['ro_ch'], cfg['ro_ch2']], pins=[0, 1], t=cfg['trig_time'])

config_corr = {
    'gen_ch': GEN_CH,
    'gen_ch2': 1,
    'ro_ch': RO_CH,
    'ro_ch2': 1,
    'freq1': 100.0,
    'freq2': 100.0,
    'pulse_len': 0.2,
    'delay': 0.1,  # Delay between pulses (µs)
    'gain': 0.5,
    'trig_time': 0.05,
    'ro_len': 0.5,
    'corr_ch0': 0,
    'corr_ch1': 1,
    'corr_len': 100
}

print("Hardware Correlator Configuration:")
print(f"- Correlating ADC channels {config_corr['corr_ch0']} and {config_corr['corr_ch1']}")
print(f"- Correlation length: {config_corr['corr_len']} samples")
print(f"- Pulse delay: {config_corr['delay']} µs")
print("\nThe correlator computes: C[τ] = Σ I(t) * I(t+τ) + Q(t) * Q(t+τ)")
print("This provides real-time cross-correlation between two channels.")

## 6. Multi-Tone Generation

You can generate multiple tones simultaneously by summing signals from different DDS channels. This enables:
- Simultaneous qubit control
- Frequency comb generation
- Parallel readout of multiple resonators

In [ ]:
class MultiToneProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        # Declare multiple generators on the same DAC channel
        # Each generator can have its own frequency
        self.declare_gen(ch=cfg['gen_ch'], nqz=1, name='tone1')
        self.declare_gen(ch=cfg['gen_ch'], nqz=1, name='tone2')
        self.declare_gen(ch=cfg['gen_ch'], nqz=1, name='tone3')
        
        self.declare_readout(ch=cfg['ro_ch'], length=cfg['ro_len'])

        # Define constant pulses for each tone
        self.add_pulse(ch=cfg['gen_ch'], name="pulse1",
                       style="const", freq=cfg['freq1'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain1'])
        
        self.add_pulse(ch=cfg['gen_ch'], name="pulse2",
                       style="const", freq=cfg['freq2'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain2'])
        
        self.add_pulse(ch=cfg['gen_ch'], name="pulse3",
                       style="const", freq=cfg['freq3'],
                       length=cfg['pulse_len'], phase=0, gain=cfg['gain3'])

        self.add_readoutconfig(ch=cfg['ro_ch'], name="my_ro",
                               freq=cfg['freq1'], gen_ch=cfg['gen_ch'])

    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'])
        
        # Play all tones simultaneously (summed on the DAC)
        # The QICK automatically sums signals from multiple generators
        self.pulse(ch=cfg['gen_ch'], name="pulse1", t=0)
        self.pulse(ch=cfg['gen_ch'], name="pulse2", t=0)
        self.pulse(ch=cfg['gen_ch'], name="pulse3", t=0)

config_multitone = {
    'gen_ch': GEN_CH,
    'ro_ch': RO_CH,
    'freq1': 95.0,   # MHz
    'freq2': 100.0,  # MHz
    'freq3': 105.0,  # MHz
    'gain1': 0.3,
    'gain2': 0.3,
    'gain3': 0.3,
    'pulse_len': 0.5,
    'trig_time': 0.05,
    'ro_len': 0.8
}

print("Multi-Tone Generation:")
print(f"- Tone 1: {config_multitone['freq1']} MHz at gain {config_multitone['gain1']}")
print(f"- Tone 2: {config_multitone['freq2']} MHz at gain {config_multitone['gain2']}")
print(f"- Tone 3: {config_multitone['freq3']} MHz at gain {config_multitone['gain3']}")
print("\nAll tones are summed and output on the same DAC channel.")
print("This is useful for:")
print("  - Simultaneous driving of multiple qubits")
print("  - Frequency-multiplexed readout")
print("  - Creating frequency combs for spectroscopy")

## 7. Advanced DSP Examples

Here are some advanced DSP techniques you can implement with the QICK.

In [ ]:
def advanced_dsp_examples():
    """Collection of advanced DSP techniques."""
    
    # Example 1: Matched Filter for Optimal SNR
    def matched_filter_coefficients(pulse_shape, num_taps=31):
        """
        Create a matched filter for a given pulse shape.
        The matched filter maximizes SNR for known signals.
        """
        if isinstance(pulse_shape, str):
            if pulse_shape == 'gaussian':
                # Gaussian pulse shape
                t = np.linspace(-2, 2, num_taps)
                coeffs = np.exp(-t**2 / 2)
            elif pulse_shape == 'square':
                coeffs = np.ones(num_taps)
            else:
                coeffs = np.random.randn(num_taps)
        else:
            coeffs = np.array(pulse_shape)
        
        # Normalize
        coeffs = coeffs / np.sum(np.abs(coeffs))
        coeffs = (coeffs / np.max(np.abs(coeffs)) * 32767).astype(int)
        return coeffs
    
    # Example 2: Notch Filter for Interference Rejection
    def notch_filter_coefficients(notch_freq_mhz, sample_rate_mhz=100, num_taps=31, Q=10):
        """
        Design a notch filter to remove a specific interference frequency.
        """
        nyquist = sample_rate_mhz / 2
        normalized_notch = notch_freq_mhz / nyquist
        
        # Simple notch using a second-order IIR converted to FIR
        b, a = signal.iirnotch(normalized_notch, Q)
        
        # Convert to FIR (truncate impulse response)
        impulse = np.zeros(num_taps * 2)
        impulse[0] = 1
        response = signal.lfilter(b, a, impulse)
        coeffs = response[:num_taps]
        
        coeffs = (coeffs / np.max(np.abs(coeffs)) * 32767).astype(int)
        return coeffs
    
    # Example 3: Hilbert Transform for Envelope Detection
    def hilbert_coefficients(num_taps=31):
        """
        Hilbert transform coefficients for IQ demodulation.
        """
        n = np.arange(num_taps) - (num_taps - 1) / 2
        n[n == 0] = 1  # Avoid division by zero
        coeffs = (1 - (-1)**n) / (np.pi * n)
        coeffs = (coeffs / np.max(np.abs(coeffs)) * 32767).astype(int)
        return coeffs
    
    print("Advanced DSP Techniques for QICK:")
    print("\n1. Matched Filters")
    gaussian_mf = matched_filter_coefficients('gaussian')
    print(f"   Gaussian matched filter taps: {gaussian_mf[:5]}...")
    
    print("\n2. Notch Filters for Interference Rejection")
    notch = notch_filter_coefficients(60, Q=30)  # Remove 60 Hz interference
    print(f"   60 Hz notch filter taps: {notch[:5]}...")
    
    print("\n3. Hilbert Transform for Envelope Detection")
    hilbert = hilbert_coefficients()
    print(f"   Hilbert filter taps: {hilbert[:5]}...")

advanced_dsp_examples()

## 8. Summary

You have learned:
- How to design and apply FIR filters for pulse shaping and signal processing
- How to dynamically tune DDS frequencies during sequences
- How to use hardware correlators for cross-channel measurements
- How to generate multi-tone signals for parallel operations
- Advanced DSP techniques like matched filtering and notch filtering

**Key takeaways:**
- FIR filters enable real-time spectral shaping on DAC outputs
- Dynamic DDS tuning allows frequency modulation within sequences
- Hardware correlators provide real-time cross-correlation at high speed
- Multi-tone generation enables parallel control and readout

**Next steps:** Proceed to [`12_Custom_Firmware_Integration.ipynb`](./12_Custom_Firmware_Integration.ipynb) to learn about adding custom Verilog modules to the QICK firmware.